---
title: "TabPFN Quickstart"
description: "Classify and regress with the hosted TabPFN API, then run locally."
icon: "bolt"
cookbookTags:
  - api
authors:
  - name: Eliott Kalfon
    linkedin: https://www.linkedin.com/in/eliott-kalfon/
---

# TabPFN Quickstart

Use TabPFN's pretrained model with scikit-learn's `fit` / `predict` workflow. Start with the hosted API; try local inference at the end.

## Setup

Install the client, local package, and example dependencies.

In [ ]:
%pip install -q tabpfn tabpfn-client scikit-learn

## Authenticate

Sign in at [Prior Labs](https://ux.priorlabs.ai), accept the license, and copy your API key into a Colab secret named `TABPFN_TOKEN`. Enable notebook access to that secret.

In [ ]:
import os
from google.colab import userdata
from tabpfn_client import set_access_token

os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
set_access_token(userdata.get('TABPFN_TOKEN'))

## Classification

Split the German credit dataset, then evaluate held-out predictions with ROC AUC and accuracy. `fit` supplies context to the pretrained model without updating its weights.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn_client import TabPFNClassifier

X, y = fetch_openml(data_id=46562, as_frame=True, return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

clf = TabPFNClassifier()
clf.fit(X_train, y_train)

prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

ROC AUC: 0.8050636232454415


Accuracy 0.7818181818181819


## Regression

Predict diabetes progression and report mean squared error, mean absolute error, and R².

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tabpfn_client import TabPFNRegressor

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42,
)

reg = TabPFNRegressor()
reg.fit(X_train, y_train)

predictions = reg.predict(X_test)
print("Mean Squared Error (MSE):", mean_squared_error(y_test, predictions))
print("Mean Absolute Error (MAE):", mean_absolute_error(y_test, predictions))
print("R-squared (R^2):", r2_score(y_test, predictions))

Mean Squared Error (MSE): 2698.666027142602
Mean Absolute Error (MAE): 40.93228039676196
R-squared (R^2): 0.5310957151096967


## Run locally

Switch the import to `tabpfn` to run inference on your machine. The first use downloads model weights; a GPU is recommended for this example.

In [ ]:
from tabpfn import TabPFNClassifier

X, y = fetch_openml(data_id=46562, as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

clf = TabPFNClassifier()  # downloads weights on first use, then runs locally
clf.fit(X_train, y_train)
print("Local ROC AUC:", roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]))

Local ROC AUC: 0.8055446237264419
